# Exact-event EB: 3-seed Validation

## 고정 검증

seed42 screen에서 통과한 exact-event Empirical-Bayes 구성을 그대로 42/777/2024에서 검증합니다. 새 피처, margin, LR/LGBM 파라미터, blend weight는 탐색하지 않습니다.

- seed42는 이미 생성된 동일 run 결과를 재사용합니다.
- 777, 2024는 동일 Stratified 5-fold로 새로 실행합니다.
- test는 읽지 않으며, 모든 exact vocabulary/EB 통계/표준화는 outer-fold train only입니다.
- 세 seed 모두 양수, 평균 +0.010 이상, 최소 +0.005 이상, 15개 fold 중 11개 이상 상승일 때만 채택합니다.

In [ ]:
from pathlib import Path
import subprocess, sys
from tqdm.auto import tqdm

ROOT = Path('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton')
BASE = ROOT / 'experiments' / 'gs' / 'notebooks' / 'exp_model_015'
RUNNER = BASE / 'common' / 'run_exact_event_eb_3seed_validation.py'
RESULT = BASE / 'result'
RUN_ID = 'exp-exact-event-eb-01'
RUN_EXPERIMENT = True
assert (RESULT / f'{RUN_ID}_seed42_summary.csv').exists(), '먼저 exp-exact-event-eb-01 seed42 screen 결과가 필요합니다.'
print({'runner': RUNNER, 'reuse_seed42': True, 'run_seeds': (777, 2024), 'test_read': False})

In [ ]:
if RUN_EXPERIMENT:
    process = subprocess.Popen([sys.executable, str(RUNNER), '--run-id', RUN_ID], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in tqdm(process.stdout, desc='exact-event EB 3-seed validation', unit='line'):
        print(line, end='')
        tail = (tail + [line])[-120:]
    if process.wait():
        raise RuntimeError('3-seed validation failed:\n' + ''.join(tail))
else:
    print('RUN_EXPERIMENT=False: existing result files only.')

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

seed_summary = pd.read_csv(RESULT / f'{RUN_ID}_3seed_summary.csv')
aggregate = pd.read_csv(RESULT / f'{RUN_ID}_3seed_aggregate.csv')
folds = pd.read_csv(RESULT / f'{RUN_ID}_3seed_fold_metrics.csv')
decision = json.loads((RESULT / f'{RUN_ID}_3seed_decision.json').read_text())
assert seed_summary.leakage_check.all() and seed_summary.nan_as_mutation_count.eq(0).all()
display(seed_summary.sort_values(['seed', 'variant']))
display(aggregate)
display(folds.pivot(index=['seed', 'fold'], columns='variant', values='macro_f1'))
folds.pivot(index=['seed', 'fold'], columns='variant', values='macro_f1').plot(marker='o', figsize=(9, 4), title='H0 vs exact-event EB: 3 seeds')
plt.ylabel('Macro F1'); plt.tight_layout(); plt.show()
print(decision)